# Faster R-CNN — UFPR-ALPR Plate Detection

Fine-tune an internet-pretrained TorchVision Faster R-CNN on the existing YOLO-format dataset. Model checkpoints, training history, and evaluation metrics are saved in Google Drive.

- Project: `/content/drive/MyDrive/UFPR_ALPR_Project`
- Original dataset: `/content/drive/MyDrive/UFPR-ALPR dataset`
- Prepared labels/images: `UFPR_ALPR_Project/yolo_dataset`

This is **plate detection only**; PARSeq remains the separate OCR model. Select a GPU runtime before training.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, glob, time, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
PROJECT = "/content/drive/MyDrive/UFPR_ALPR_Project"
DATASET = f"{PROJECT}/yolo_dataset"
IMAGE_DIR, LABEL_DIR = f"{DATASET}/images", f"{DATASET}/labels"
OUTPUT_DIR = f"{PROJECT}/faster_rcnn"
CKPT_DIR = f"{OUTPUT_DIR}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
BEST_PATH, LAST_PATH = f"{CKPT_DIR}/fasterrcnn_plate_best.pth", f"{CKPT_DIR}/fasterrcnn_plate_last.pth"
HISTORY_PATH, METRICS_PATH = f"{OUTPUT_DIR}/training_history.csv", f"{OUTPUT_DIR}/test_metrics.json"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
for split in ("train", "val", "test"):
    print(split, "images:", len(glob.glob(f"{IMAGE_DIR}/{split}/*")) if os.path.isdir(f"{IMAGE_DIR}/{split}") else "MISSING",
          "| labels:", len(glob.glob(f"{LABEL_DIR}/{split}/*.txt")) if os.path.isdir(f"{LABEL_DIR}/{split}") else "MISSING")
assert os.path.isdir(f"{IMAGE_DIR}/train") and os.path.isdir(f"{LABEL_DIR}/train"), "Expected yolo_dataset/images/train and labels/train"

## 1. Dataset

Uses the prepared YOLO labels (`class x_center y_center width height`) and converts them to pixel boxes. Faster R-CNN uses class `1` for the license plate; `0` is background.

In [ ]:
class PlateDataset(Dataset):
    def __init__(self, split):
        self.img_dir, self.lbl_dir = f"{IMAGE_DIR}/{split}", f"{LABEL_DIR}/{split}"
        self.paths = sorted(p for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.webp")
                            for p in glob.glob(os.path.join(self.img_dir, ext)))
        if not self.paths: raise FileNotFoundError(f"No images found: {self.img_dir}")

    def __len__(self): return len(self.paths)

    def __getitem__(self, i):
        path = self.paths[i]
        im = Image.open(path).convert("RGB")
        w, h = im.size
        lp = os.path.join(self.lbl_dir, os.path.splitext(os.path.basename(path))[0]+".txt")
        boxes, labels = [], []
        if os.path.isfile(lp):
            with open(lp) as f:
                for line in f:
                    v = line.split()
                    if len(v) < 5: continue
                    _, xc, yc, bw, bh = map(float, v[:5])
                    b = [max(0,(xc-bw/2)*w), max(0,(yc-bh/2)*h),
                         min(w,(xc+bw/2)*w), min(h,(yc+bh/2)*h)]
                    if b[2] > b[0] and b[3] > b[1]:
                        boxes.append(b); labels.append(1)
        target = {"boxes": torch.tensor(boxes, dtype=torch.float32).reshape(-1,4),
                  "labels": torch.tensor(labels, dtype=torch.int64),
                  "image_id": torch.tensor([i]),
                  "area": torch.tensor([(b[2]-b[0])*(b[3]-b[1]) for b in boxes], dtype=torch.float32),
                  "iscrowd": torch.zeros(len(boxes), dtype=torch.int64)}
        return to_tensor(im), target

def collate_fn(batch): return tuple(zip(*batch))

val_split = "val" if os.path.isdir(f"{IMAGE_DIR}/val") else "validation"
train_ds, val_ds, test_ds = PlateDataset("train"), PlateDataset(val_split), PlateDataset("test")
BATCH_SIZE = 2 if torch.cuda.is_available() else 1
WORKERS = 2 if torch.cuda.is_available() else 0
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=WORKERS, collate_fn=collate_fn)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=WORKERS, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=WORKERS, collate_fn=collate_fn)
print(f"Train={len(train_ds)} | Validation={len(val_ds)} | Test={len(test_ds)} | Batch={BATCH_SIZE}")

## 2. Inspect a training example

In [ ]:
im, target = train_ds[0]
plt.figure(figsize=(12,6)); plt.imshow(im.permute(1,2,0).numpy())
ax = plt.gca()
for x1,y1,x2,y2 in target["boxes"].numpy():
    ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color="lime",linewidth=2))
plt.title("Ground-truth license plate boxes"); plt.axis("off"); plt.show()

## 3. Load pretrained Faster R-CNN

TorchVision downloads COCO-pretrained ResNet-50-FPN weights on first use and replaces the prediction head for one class. This follows the official TorchVision fine-tuning approach.

In [ ]:
def build_model():
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    n = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(n, 2)  # plate + background
    return model

model = build_model().to(device)
optimizer = torch.optim.SGD([p for p in model.parameters() if p.requires_grad],
                            lr=0.005, momentum=0.9, weight_decay=0.0005)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
EPOCHS = 10
start_epoch, best_val_loss, history = 0, float("inf"), []
if os.path.isfile(LAST_PATH):
    ck = torch.load(LAST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ck["model_state"]); optimizer.load_state_dict(ck["optimizer_state"])
    if "scheduler_state" in ck: scheduler.load_state_dict(ck["scheduler_state"])
    start_epoch = ck["epoch"] + 1
    best_val_loss, history = ck.get("best_val_loss", float("inf")), ck.get("history", [])
    print("Resuming from epoch", start_epoch)
else:
    print("Starting from COCO-pretrained weights.")
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Train, validate, and save to Drive
A `last` checkpoint is saved after each epoch for resuming after a Colab reset. The `best` checkpoint is selected by validation loss.

In [ ]:
!pip install -q tqdm

In [ ]:

from tqdm.auto import tqdm
import time
import numpy as np
import pandas as pd
import torch

def validation_loss():
    model.train()  # TorchVision needs training mode to return loss dictionaries
    total, count = 0.0, 0

    pbar = tqdm(val_loader, desc="Validation", leave=False)
    with torch.no_grad():
        for imgs, tgts in pbar:
            imgs = [x.to(device) for x in imgs]
            tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                loss = sum(model(imgs, tgts).values())

            total += loss.item()
            count += 1
            pbar.set_postfix(loss=f"{total / count:.4f}")

    return total / count if count else float("inf")


for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss, count = 0.0, 0
    t0 = time.time()

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS} Training",
        unit="batch"
    )

    for imgs, tgts in pbar:
        imgs = [x.to(device) for x in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in tgts]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            loss = sum(model(imgs, tgts).values())

        if not torch.isfinite(loss):
            pbar.set_postfix(status="Skipped non-finite loss")
            continue

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        count += 1

        elapsed = time.time() - t0
        avg_time = elapsed / max(1, pbar.n + 1)
        remaining = avg_time * (len(train_loader) - pbar.n - 1)

        pbar.set_postfix(
            loss=f"{total_loss / count:.4f}",
            elapsed=f"{elapsed / 60:.1f}m",
            ETA=f"{remaining / 60:.1f}m"
        )

    train_loss = total_loss / count if count else float("inf")

    print(f"\nEpoch {epoch + 1}: Training finished. Starting validation...")
    val_loss_value = validation_loss()
    scheduler.step()

    epoch_seconds = time.time() - t0

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss_value,
        "seconds": epoch_seconds
    })

    if val_loss_value < best_val_loss:
        best_val_loss = val_loss_value
        torch.save({
            "model_state": model.state_dict(),
            "epoch": epoch,
            "best_val_loss": best_val_loss
        }, BEST_PATH)
        status = "BEST MODEL UPDATED"
    else:
        status = "No improvement"

    torch.save({
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "epoch": epoch,
        "best_val_loss": best_val_loss,
        "history": history
    }, LAST_PATH)

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} COMPLETE | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Loss: {val_loss_value:.4f} | "
        f"Time: {epoch_seconds / 60:.1f} min | {status}"
    )
    print(f"Epochs remaining: {EPOCHS - epoch - 1}")

print("\nTRAINING FINISHED")
print("Best checkpoint:", BEST_PATH)
print("Resume checkpoint:", LAST_PATH)
print("Training history:", HISTORY_PATH)


## 5. Training curve

In [ ]:
hist = pd.read_csv(HISTORY_PATH)
display(hist.tail())
plt.figure(figsize=(8,4))
plt.plot(hist.epoch,hist.train_loss,marker="o",label="Train loss")
plt.plot(hist.epoch,hist.val_loss,marker="o",label="Validation loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Faster R-CNN training history")
plt.grid(True); plt.legend(); plt.show()

## 6. Test evaluation

Computes precision, recall, F1 and mean matched IoU at IoU ≥ 0.50 with a greedy one-to-one match. These are custom metrics, **not COCO mAP**.

In [ ]:
ck = torch.load(BEST_PATH, map_location=device, weights_only=False)
model.load_state_dict(ck["model_state"]); model.to(device).eval()

def ious_one(box, boxes):
    if len(boxes)==0: return np.zeros(0)
    xx1=np.maximum(box[0],boxes[:,0]); yy1=np.maximum(box[1],boxes[:,1])
    xx2=np.minimum(box[2],boxes[:,2]); yy2=np.minimum(box[3],boxes[:,3])
    inter=np.maximum(0,xx2-xx1)*np.maximum(0,yy2-yy1)
    a=max(0,box[2]-box[0])*max(0,box[3]-box[1])
    b=np.maximum(0,boxes[:,2]-boxes[:,0])*np.maximum(0,boxes[:,3]-boxes[:,1])
    return inter/np.maximum(a+b-inter,1e-9)

TP=FP=FN=0; matched=[]; infer_time=0.0
SCORE_THR, IOU_THR = 0.5, 0.5
with torch.inference_mode():
    for imgs,tgts in test_loader:
        x=imgs[0].to(device); gt=tgts[0]["boxes"].numpy()
        t0=time.perf_counter(); out=model([x])[0]
        if device.type=="cuda": torch.cuda.synchronize()
        infer_time += time.perf_counter()-t0
        scores=out["scores"].cpu().numpy()
        boxes=out["boxes"].cpu().numpy()
        keep=np.where(scores>=SCORE_THR)[0]
        keep=keep[np.argsort(-scores[keep])]
        used=set()
        for j in keep:
            ov=ious_one(boxes[j],gt)
            if len(ov) and ov.max()>=IOU_THR and int(ov.argmax()) not in used:
                k=int(ov.argmax()); used.add(k); TP+=1; matched.append(float(ov[k]))
            else: FP+=1
        FN += len(gt)-len(used)

p=TP/(TP+FP) if TP+FP else 0
r=TP/(TP+FN) if TP+FN else 0
metrics={"model":"Faster R-CNN ResNet50-FPN","test_images":len(test_ds),
         "score_threshold":SCORE_THR,"iou_threshold":IOU_THR,
         "TP":TP,"FP":FP,"FN":FN,"precision":p,"recall":r,
         "F1":2*p*r/(p+r) if p+r else 0,
         "mean_matched_IoU":float(np.mean(matched)) if matched else 0,
         "total_inference_seconds":infer_time,
         "average_inference_ms_per_image":1000*infer_time/max(1,len(test_ds))}
with open(METRICS_PATH,"w") as f: json.dump(metrics,f,indent=4)
print("\n"+"="*52+"\n FASTER R-CNN — TEST REPORT\n"+"="*52)
for k,v in metrics.items(): print(f"{k:<34}: {v:.4f}" if isinstance(v,float) else f"{k:<34}: {v}")
print("Saved:",METRICS_PATH)

In [ ]:
!pip install -q pytorch-lightning timm einops

## 7. Visual demo: predictions vs ground truth
Green boxes are ground truth; red boxes are model predictions.

In [ ]:
def show_prediction(index=0, threshold=0.5):
    im, target = test_ds[index]
    model.eval()
    with torch.inference_mode(): pred=model([im.to(device)])[0]
    arr=(im.permute(1,2,0).numpy()*255).astype(np.uint8)
    fig,ax=plt.subplots(figsize=(12,6)); ax.imshow(arr)
    for x1,y1,x2,y2 in target["boxes"].numpy():
        ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color="lime",linewidth=2))
    b=pred["boxes"].cpu().numpy(); s=pred["scores"].cpu().numpy()
    for box,score in zip(b,s):
        if score<threshold: continue
        x1,y1,x2,y2=box
        ax.add_patch(plt.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,color="red",linewidth=2))
        ax.text(x1,max(0,y1-5),f"Plate {score:.2f}",color="white",
                bbox=dict(facecolor="red",alpha=.8,pad=2))
    ax.set_title(f"Test image {index}: green=ground truth, red=prediction")
    ax.axis("off"); plt.show()

for i in range(min(3,len(test_ds))): show_prediction(i)

In [ ]:

# ============================================================
# CELL 0: Install dependencies for YOLOv8 + fine-tuned PARSeq
# Run this cell first after starting/restarting Colab.
# ============================================================

%pip install -q --upgrade pip

%pip install -q \
    ultralytics \
    torch torchvision torchaudio \
    timm einops \
    pytorch-lightning \
    torchmetrics \
    omegaconf hydra-core \
    lmdb \
    opencv-python-headless \
    pillow \
    numpy \
    pandas \
    matplotlib \
    tqdm \
    scikit-learn \
    scipy \
    tensorboard

print("Installation cell completed.")
print("If Colab asks for a runtime restart, restart and run Cell 0 again only if imports fail.")


In [ ]:

# CELL 1: Load fine-tuned PARSeq OCR

!pip -q install timm einops pytorch-lightning torchmetrics omegaconf hydra-core ultralytics

import os
import torch
from torchvision.transforms import functional as TF
from PIL import Image

from google.colab import drive
drive.mount("/content/drive")

PROJECT_DIR = "/content/drive/MyDrive/UFPR_ALPR_Project"
PARSEQ_PATH = os.path.join(
    PROJECT_DIR, "parseq_ocr", "parseq_ufpr_best.pt"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

if not os.path.isfile(PARSEQ_PATH):
    raise FileNotFoundError(
        f"PARSeq checkpoint not found:\n{PARSEQ_PATH}"
    )

# Create the original PARSeq architecture
parseq_model = torch.hub.load(
    "baudm/parseq",
    "parseq",
    pretrained=True,
    trust_repo=True
)

# Load fine-tuned weights
checkpoint = torch.load(
    PARSEQ_PATH,
    map_location=DEVICE,
    weights_only=True
)

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    parseq_model.load_state_dict(checkpoint["model_state_dict"])
else:
    parseq_model.load_state_dict(checkpoint)

parseq_model = parseq_model.to(DEVICE).eval()

# Use PARSeq's expected image preprocessing
from strhub.data.module import SceneTextDataModule
parseq_transform = SceneTextDataModule.get_transform(
    parseq_model.hparams.img_size
)

print("Fine-tuned PARSeq loaded successfully.")
print("Checkpoint:", PARSEQ_PATH)


In [ ]:

# ============================================================
# CELL 2: Faster R-CNN + fine-tuned PARSeq — 5-image demo
# ============================================================

import os
import glob
import re
import random
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision.transforms.functional import to_tensor
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

PROJECT_DIR = "/content/drive/MyDrive/UFPR_ALPR_Project"
DATASET_DIR = f"{PROJECT_DIR}/yolo_dataset"
TEST_IMAGES = f"{DATASET_DIR}/images/test"
CHECKPOINT_PATH = (
    f"{PROJECT_DIR}/faster_rcnn/checkpoints/"
    "fasterrcnn_plate_best.pth"
)
SOURCE_DIR = "/content/drive/MyDrive/UFPR-ALPR dataset"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Check files and PARSeq ----------
if not os.path.isfile(CHECKPOINT_PATH):
    raise FileNotFoundError(f"Faster R-CNN checkpoint not found:\n{CHECKPOINT_PATH}")

if "parseq_model" not in globals() or "parseq_transform" not in globals():
    raise RuntimeError("Run your PARSeq loading cell first.")

# ---------- Rebuild Faster R-CNN architecture ----------
def build_faster_rcnn():
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, 2)
    return model

detector = build_faster_rcnn()

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

# Matches the checkpoint format used by your training notebook
state_dict = (
    checkpoint["model_state"]
    if isinstance(checkpoint, dict) and "model_state" in checkpoint
    else checkpoint
)
detector.load_state_dict(state_dict)
detector = detector.to(DEVICE).eval()

parseq_model = parseq_model.to(DEVICE).eval()

print("Faster R-CNN and fine-tuned PARSeq loaded.")
print("Device:", DEVICE)

# ---------- Original annotation index ----------
annotation_index = {}

for split in ("training", "validation", "testing"):
    folder = os.path.join(SOURCE_DIR, split)
    if os.path.isdir(folder):
        for txt_path in glob.glob(
            os.path.join(folder, "**", "*.txt"), recursive=True
        ):
            stem = os.path.splitext(os.path.basename(txt_path))[0]
            annotation_index[stem] = txt_path

def get_actual_plate(image_path):
    stem = os.path.splitext(os.path.basename(image_path))[0]
    candidates = [stem, stem.split("__", 1)[-1]]

    for candidate in candidates:
        txt_path = annotation_index.get(candidate)
        if not txt_path:
            continue

        with open(txt_path, encoding="utf-8") as f:
            for line in f:
                if line.strip().lower().startswith("plate:"):
                    return line.split(":", 1)[1].strip().upper()

    return "NOT FOUND"

def normalize_plate(text):
    return re.sub(r"[^A-Z0-9]", "", str(text).upper())

def read_plate(crop_bgr):
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(crop_rgb)
    tensor = parseq_transform(pil_image).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        logits = parseq_model(tensor)
        decoded = parseq_model.tokenizer.decode(logits.softmax(dim=-1))

    texts = decoded[0]
    prediction = texts[0] if isinstance(texts, (list, tuple)) else texts
    return str(prediction).strip().upper()

# ---------- Select five test images ----------
image_paths = []
for ext in ("*.png", "*.jpg", "*.jpeg", "*.bmp"):
    image_paths.extend(glob.glob(os.path.join(TEST_IMAGES, ext)))

image_paths = sorted(set(image_paths))
if not image_paths:
    raise FileNotFoundError(f"No test images found in {TEST_IMAGES}")

random.seed(42)
selected = random.sample(image_paths, min(5, len(image_paths)))
results = []

fig, axes = plt.subplots(len(selected), 2, figsize=(12, 4 * len(selected)))
if len(selected) == 1:
    axes = [axes]

# ---------- Detection + OCR ----------
for i, image_path in enumerate(selected):
    image_bgr = cv2.imread(image_path)
    if image_bgr is None:
        continue

    h, w = image_bgr.shape[:2]
    image_tensor = to_tensor(
        Image.fromarray(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    ).to(DEVICE)

    with torch.inference_mode():
        output = detector([image_tensor])[0]

    # Keep plate-class predictions above this confidence threshold
    scores = output["scores"]
    boxes = output["boxes"]
    labels = output["labels"]
    keep = (scores >= 0.50) & (labels == 1)

    actual = get_actual_plate(image_path)
    predicted = "NO PLATE DETECTED"
    crop = None

    if keep.any():
        candidate_indices = torch.where(keep)[0]
        best_idx = candidate_indices[
            scores[candidate_indices].argmax()
        ].item()

        x1, y1, x2, y2 = map(
            int, boxes[best_idx].detach().cpu().tolist()
        )
        x1, x2 = max(0, x1), min(w, x2)
        y1, y2 = max(0, y1), min(h, y2)

        if x2 > x1 and y2 > y1:
            crop = image_bgr[y1:y2, x1:x2]
            predicted = read_plate(crop)

            cv2.rectangle(
                image_bgr, (x1, y1), (x2, y2), (0, 255, 0), 2
            )

    correct = (
        actual != "NOT FOUND"
        and normalize_plate(actual) == normalize_plate(predicted)
    )

    results.append({
        "image": os.path.basename(image_path),
        "actual": actual,
        "predicted": predicted,
        "exact_match": correct
    })

    axes[i][0].imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(
        f"Image {i+1}\nActual: {actual}\nPredicted: {predicted}"
    )
    axes[i][0].axis("off")

    if crop is not None:
        axes[i][1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
        axes[i][1].set_title(
            f"Detected plate | {'CORRECT' if correct else 'INCORRECT'}"
        )
    else:
        axes[i][1].text(0.5, 0.5, "No plate detected", ha="center", va="center")
        axes[i][1].set_title("Detected plate")

    axes[i][1].axis("off")

plt.tight_layout()
plt.show()

# ---------- Print results ----------
print("\n========== FASTER R-CNN + PARSEQ RESULTS ==========")
for row in results:
    status = "CORRECT" if row["exact_match"] else "INCORRECT"
    print(
        f"{row['image']} | Actual: {row['actual']} | "
        f"Predicted: {row['predicted']} | {status}"
    )

valid = [r for r in results if r["actual"] != "NOT FOUND"]
if valid:
    correct_count = sum(r["exact_match"] for r in valid)
    print(
        f"\nEnd-to-end exact plate accuracy: "
        f"{correct_count}/{len(valid)} "
        f"({100 * correct_count / len(valid):.2f}%)"
    )
else:
    print("\nActual plate text was not found for these image filenames.")
    print("Check the original annotation filenames and matching logic.")


## 8. Final files

All results are saved under `UFPR_ALPR_Project/faster_rcnn/`:
- `checkpoints/fasterrcnn_plate_best.pth` — best model
- `checkpoints/fasterrcnn_plate_last.pth` — resumable training checkpoint
- `training_history.csv` — epoch losses
- `test_metrics.json` — test metrics

## References
- [Official TorchVision Faster R-CNN docs](https://docs.pytorch.org/vision/main/models/faster_rcnn.html)
- [Official TorchVision fine-tuning tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html)
- [UFPR-ALPR dataset description](https://web.inf.ufpr.br/vri/databases/ufpr-alpr/)

**Assumption:** the prepared data has `yolo_dataset/images/{train,val,test}` and matching `labels/{train,val,test}` folders. The notebook reuses those YOLO-format annotations. If your folders use different split names, update the paths before running.